In [31]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal, Annotated
from langchain_core.messages import SystemMessage, HumanMessage

In [32]:
generator_llm =  ChatOpenAI(model='gpt-4o')
evaluator_llm =  ChatOpenAI(model='gpt-4o-mini')
optimizer_llm = ChatOpenAI(model='gpt-4o')

In [33]:
from pydantic import BaseModel, Field
class tweetEvaluation(BaseModel):
    evaluation: Literal["approve", "need_improvment"] = Field(...,  description="final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet")

In [34]:
from pydantic import BaseModel, Field
class tweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet")


In [35]:
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str
    iteration: int
    maz_iteration: int


In [36]:
def generate_tweet(state:TweetState):
    message = [
        SystemMessage(content= "You are a funny and clever Twitter/x influencer"),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]
    response =generator_llm.invoke(message).content

    return {"tweet": response}

In [37]:
def evaluate_tweet(state: TweetState):
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state["tweet"]}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]
    response = structured_evaluator_llm.invoke(messages)
    return {"evaluation": response.evaluation, "feedback": response.feedback}


In [38]:

def optimize_tweet(state: TweetState):

    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state["feedback"]}"

Topic: "{state["topic"]}"
Original Tweet:
{state["tweet"]}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]
    response = optimizer_llm.invoke(messages).content
    iteration = state["iteration"] + 1

    return {"tweet": response, "iteration": iteration}



In [39]:
def route_evaluation(state: TweetState):
    if state["evaluation"] == "approved" or state["iteration"] >= state["maz_iteration"]:
        return "approved"
    else:
        return "needs_improvement"


In [40]:
graph = StateGraph(TweetState)
graph.add_node("generate", generate_tweet)
graph.add_node("evaluate", evaluate_tweet)
graph.add_node("optimize", optimize_tweet)
graph.add_edge(START, "generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges("evaluate", route_evaluation, {"approved": END, "needs_improvement": "optimize"})
graph.add_edge("optimize", "evaluate")
workflow = graph.compile()


In [41]:
inital_state ={
    'topic': "Indian Railway",
    "iteration": 1,
    "maz_iteration": 5
}
workflow.invoke(inital_state)

{'topic': 'Indian Railway',
 'tweet': 'Indian Railways: the ultimate sleep dojo! Master the art of waking up five minutes before your stop—no alarm needed. These trains should start giving out black belts in snooze. 🥋😴🚉 #MasterCommuters #TrainLife',
 'evaluation': 'approve',
 'feedback': "This tweet showcases a fresh perspective on the experience of traveling on Indian Railways, tapping into the relatable struggle of waking up just in time before disembarking. Its humor effectively captures that shared commuter experience, and the clever metaphor of a 'sleep dojo' adds a nice touch that enhances originality. The punchiness of the tweet is quite good, as it is concise and contains engaging imagery. With the inclusion of emojis and relevant hashtags, the tweet is well-formatted for Twitter's style and has strong potential for virality among commuters who can relate to this experience.",
 'iteration': 5,
 'maz_iteration': 5}